In [ ]:
# imports

using JuMP
using SCIP
import MathOptInterface as MOI
using PrettyTables
include("util.jl")

In [ ]:
# Файл с входными данными

input_files_folder = "input"
# input_file_name = "input_smaller.txt"
input_file_name = "input_current.txt"

In [ ]:
# Параметры модели Банистера
k1 = 1
k2 = 2
τ1 = 42
τ2 = 7
α = 0.5

In [ ]:
# Зачитывание входных данных

function read_input(file_path::String, validate::Bool)
    lines = readlines(file_path)

    lines = filter(line -> !isempty(line) && !startswith(strip(line), "#"), lines)
    lines = map(line -> split(line, "#")[1] |> strip, lines)

    idx = 1

    D = parse(Int, lines[idx]); idx += 1
    D_ = parse.(Int, split(lines[idx])); idx += 1
    D_star = parse.(Int, split(lines[idx])); idx += 1
    P = parse(Int, lines[idx]); idx += 1
        Player_names = split(lines[idx]); idx += 1
    S = parse(Int, lines[idx]); idx += 1
        Skill_names = split(lines[idx]); idx += 1
    T = parse(Int, lines[idx]); idx += 1
        Tactic_names = split(lines[idx]); idx += 1
    E = parse(Int, lines[idx]); idx += 1
        Exercise_names = split(lines[idx]); idx += 1

    a0 = parse.(Int, split(lines[idx])); idx += 1
    b0 = [parse.(Int, split(lines[idx + i - 1])) for i in 1:S]; idx += S
    c0 = [parse.(Int, split(lines[idx + i - 1])) for i in 1:T]; idx += T
    h = [parse.(Int, split(lines[idx + i - 1])) for i in 1:P]; idx += P
    w = parse.(Int, split(lines[idx])); idx += 1
    b = [parse.(Int, split(lines[idx + i - 1])) for i in 1:S]; idx += S
    c = [parse.(Int, split(lines[idx + i - 1])) for i in 1:T]; idx += T
    v = parse.(Int, split(lines[idx])); idx += 1
    V = parse.(Int, split(lines[idx])); idx += 1

    B_star = Array{Int, 3}(undef, S, P, length(D_star))
    for d in 1:length(D_star)
        for s in 1:S
            B_star[s, :, d] = parse.(Int, split(lines[idx])); idx += 1
        end
    end

    C_star = Array{Int, 3}(undef, T, P, length(D_star))
    for d in 1:length(D_star)
        for t in 1:T
            C_star[t, :, d] = parse.(Int, split(lines[idx])); idx += 1
        end
    end

    # input validation
    @assert maximum(D_) <= D
    @assert minimum(D_) >= 1
    @assert maximum(D_star) <= D
    @assert minimum(D_star) >= 1
    @assert length(Player_names) == P
    @assert length(Skill_names) == S
    @assert length(Tactic_names) == T
    @assert length(Exercise_names) == E
    @assert length(a0) == P
    @assert size(b0) == (S,) && size(b0[1]) == (P,)
    @assert size(c0) == (T,) && size(c0[1]) == (P,)
    @assert size(h) == (P,) && size(h[1]) == (D,)
    @assert length(w) == E
    @assert size(b) == (S,) && size(b[1]) == (E,)
    @assert size(c) == (T,) && size(c[1]) == (E,)
    @assert length(v) == E
    @assert length(V) == D
    @assert size(B_star) == (S, P, length(D_star))
    @assert size(C_star) == (T, P, length(D_star))

    return D, D_, D_star, P, Player_names, S, Skill_names, T, Tactic_names, E, Exercise_names, a0, b0, c0, h, w, b, c, v, V, B_star, C_star
end

function δ1(d_::Int, d::Int)
	return k1*ℯ^(-(d-d_)/τ1)
end

function δ2(d_::Int, d::Int)
	return k2*ℯ^(-(d-d_)/τ2)
end

function β(s::Int, p::Int, d::Int)
	return 1
end

function γ(t::Int, d::Int)
	return 1
end

function ρ(t::Int)
	return 2
end

file_path = input_files_folder * "/" * input_file_name
D, D_, D_star, P, Player_names, S, Skill_names, T, Tactic_names, E, Exercise_names, a0, b0, c0, h, w, b, c, v, V, B_star, C_star = read_input(file_path, false)

In [ ]:
# Модель
model = Model(SCIP.Optimizer)

In [ ]:
# Переменные
@variable(model, x[1:E, 1:length(D_)], Bin) #>= 0, Int)
@variable(model, y[1:S, 1:P, 1:length(D_star)], Bin)
@variable(model, z[1:T, 1:P, 1:length(D_star)], Bin)
@variable(model, Z[1:T, 1:length(D_star)], Bin)
@variable(model, A_min[1:length(D_star)])

In [ ]:
# Вспомогательные функции и компоненты целевой функции
function A(p::Int, d::Int)
	result = a0[p]
	result += sum(δ1(D_[d_], d)*h[p][d_]*(sum(w[e]*x[e, d_] for e in 1:E)) for d_ in 1:length(D_) if D_[d_] <= d)
	result -= sum(δ2(D_[d_], d)*h[p][d_]*(sum(w[e]*x[e, d_] for e in 1:E)) for d_ in 1:length(D_) if D_[d_] <= d)
	return result
end

function A_avg(d::Int)
	return sum(A(p, d) for p in 1:P)/P
end

function A()
	return sum(α*A_min[d_]+(1-α)*A_avg(D_star[d_]) for d_ in 1:length(D_star))
end

function B(s::Int, p::Int, d::Int)
	result = b0[s][p]
	result += sum(h[p][d_]*sum(b[s][e]*x[e, d_] for e in 1:E) for d_ in 1:length(D_) if D_[d_] <= d)
	return result
end

function B()
	return sum(sum(sum(β(s, p, d_)*y[s, p, d_] for s in 1:S) for p in 1:P) for d_ in 1:length(D_star))
end

function C(t::Int, p::Int, d::Int)
	result = c0[t][p]
	result += sum(h[p][d_]*sum(c[t][e]*x[e, d_] for e in 1:E) for d_ in 1:length(D_) if D_[d_] <= d)
end

function C()
	return sum(sum(γ(t, d_)*Z[t, d_] for t in 1:T) for d_ in 1:length(D_star))
end

In [ ]:
# Ограничения
@constraint(model, [d_=1:length(D_)], sum(v[e]*x[e, d_] for e in 1:E) <= V[d_])
@constraint(model, [s=1:S, p=1:P, d_=1:length(D_star)], B(s, p, D_star[d_]) + B_star[s, p, d_]*(1-y[s, p, d_]) >= B_star[s, p, d_])
@constraint(model, [t=1:T, p=1:P, d_=1:length(D_star)], C(t, p, D_star[d_]) + C_star[t, p, d_]*(1-z[t, p, d_]) >= C_star[t, p, d_])
@constraint(model, [t=1:T, d_=1:length(D_star)], sum(z[t, p, d_] for p in 1:P) >= ρ(t)*Z[t, d_])
@constraint(model, [d_=1:length(D_star), p=1:P], A_min[d_] <= A(p, D_star[d_]))

In [ ]:
# Высчитывание идеальной точки: A

@objective(model, Max, A())

# HiGHS
# set_optimizer_attribute(model, "threads", Sys.CPU_THREADS)
# set_optimizer_attribute(model, "mip_rel_gap", 0.01)
# set_optimizer_attribute(model, "time_limit", 1200.0) # 20 minutes

# SCIP
set_optimizer_attribute(model, "limits/gap", 0.01)
set_optimizer_attribute(model, "limits/time", 1200.0) # 20 minutes

optimize!(model)
A_optimal = value(A())
println(termination_status(model))
if termination_status(model) == TIME_LIMIT
    gap = relative_gap(model)
    println("Relative gap: $(gap * 100)%")
end

In [ ]:
# Высчитывание идеальной точки: B

@objective(model, Max, B())

problem_name = "ideal_B"
mps_path = problem_name * ".mps"
sol_path = problem_name * ".sol"
write_to_file(model, mps_path)
println("Model written to: $mps_path")

run(`scip -c "read $mps_path" -c "set parallel maxnthreads 12" -c "set presolving emphasis aggressive" -c "set separating emphasis aggressive" -c "set heuristics emphasis aggressive" -c "set limits gap 0.01" -c "set limits time 3600" -c "concurrentopt" -c "write solution $sol_path" -c "quit"`)
println("SCIP solve complete")

obj_val, var_vals = parse_scip_solution(sol_path)

B_optimal = obj_val

In [ ]:
# Высчитывание идеальной точки: C

@objective(model, Max, C())

# HiGHS
# set_optimizer_attribute(model, "threads", Sys.CPU_THREADS)
# set_optimizer_attribute(model, "mip_rel_gap", 0.01)
# set_optimizer_attribute(model, "time_limit", 1200.0) # 20 minutes

# SCIP
set_optimizer_attribute(model, "limits/gap", 0.01)
set_optimizer_attribute(model, "limits/time", 1200.0) # 20 minutes

optimize!(model)
C_optimal = value(C())
println(termination_status(model))
if termination_status(model) == TIME_LIMIT
    gap = relative_gap(model)
    println("Relative gap: $(gap * 100)%")
end

In [ ]:
B_optimal = 400

In [ ]:
ideal_point = [3654.73, 400.0, 20.0]

In [ ]:
# Сохранение идеальной точки, определение функций нормировки

ideal_point = [A_optimal, B_optimal, C_optimal]

println("Ideal point: $ideal_point")

A_norm = () -> A()/A_optimal
B_norm = () -> B()/B_optimal
C_norm = () -> C()/C_optimal

In [ ]:
# Решение задачи с равными весами

@objective(model, Max, A_norm() + B_norm() + C_norm())

problem_name = "033A_033B_033C"
mps_path = problem_name * ".mps"
sol_path = problem_name * ".sol"
write_to_file(model, mps_path)
println("Model written to: $mps_path")

run(`scip -c "read $mps_path" -c "set parallel maxnthreads 12" -c "set presolving emphasis aggressive" -c "set separating emphasis aggressive" -c "set heuristics emphasis aggressive" -c "set limits gap 0.01" -c "set limits time 3600" -c "concurrentopt" -c "write solution $sol_path" -c "quit"`)
# run(`scip -c "read $mps_path" -c "set parallel maxnthreads 12" -c "concurrentopt" -c "write solution $sol_path" -c "quit"`)
println("SCIP solve complete")

obj_val, var_vals = parse_scip_solution(sol_path)